# Google Trends: Pull Data and Basic EDA

**What this notebook does**
1. Pulls weekly Google Trends data for 137 fashion keywords across two overlapping windows (2017-06 to present), stitches them into a single ~9-year series per keyword, and saves the raw table.
2. Cleans the data: drops keywords with more than 30% zero weeks, flags weeks where many unrelated keywords spike together.
3. Runs basic EDA: summary statistics, category trends, seasonality, rising/falling keywords, correlations, the distribution of 4-week growth ratios, STL decomposition, class balance, and a pull-stability check.

**Notes on comparability**
Each keyword is pulled alone (no anchor term), so Google scales it to its own 0–100 range. Values are comparable *over time within a keyword*, but **not between keywords**. Do not compare absolute levels across styles.

**Before you run in Colab**
- Upload `google_trends_keywords.xlsx` to the session (folder icon → upload button).
- Set `TEST_RUN = True` for a quick 11-keyword check first, then `False` for the full pull.
- A full pull makes ~274 requests at 8-second intervals: allow at least 40 minutes.
- Download generated CSVs (and `raw_pulls/`) before the session ends — Colab storage is temporary.


In [ ]:
!pip install pytrends openpyxl -q


In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pytrends.request import TrendReq

sns.set_theme(style="whitegrid")

## 1. Settings

- **GEO**: keep the same region for every pull.
- **RECENT_WINDOW / OLD_WINDOW**: Google only returns weekly data for ranges up to about 5 years, so we pull two windows that overlap by ~28 weeks. The older window is rescaled to match the recent one over the overlap.
- **TEST_RUN**: set to `True` for a quick check (~11 keywords), then `False` for the full pull.
- **SLEEP_SECONDS**: pause between requests to avoid HTTP 429 rate-limit errors.
- **SAVE_DIR**: each pulled keyword is cached here; re-runs skip already-cached keywords.


In [ ]:
GEO = ""                                 # same region for every pull ("" = worldwide)
RECENT_WINDOW = "today 5-y"              # weekly, last 5 years
OLD_WINDOW = "2017-06-04 2022-04-30"     # weekly, earlier years; overlaps the recent window by ~28 weeks
TEST_RUN = False                         # True = quick check with 11 keywords
SLEEP_SECONDS = 8                        # pause between requests (avoids rate limits)
SAVE_DIR = "raw_pulls"                   # every pulled keyword is saved here

## 2. Load keyword list

Reads `google_trends_keywords.xlsx` (sheet `Keywords`, columns `Keyword` and `Category`). The current list has 137 keywords.


In [ ]:
kw_df = pd.read_excel("google_trends_keywords.xlsx", sheet_name="Keywords")

if TEST_RUN:
    kw_df = kw_df.groupby("Category").head(1)   # one keyword per category

keywords = list(kw_df["Keyword"])
category = dict(zip(kw_df["Keyword"], kw_df["Category"]))   # keyword -> category
len(keywords)

## 3. Pull the data

Each keyword is requested alone, scaled to its own 0–100 range. Values are comparable over time within a keyword, but **not between keywords**. This is fine for forecasting each style and for the 4-week "rising" ratio; do not compare absolute levels across styles.

Every result is cached in `raw_pulls/`. Re-running the cell skips already-cached keywords and only requests failed or new ones.


In [ ]:
import os
os.makedirs(SAVE_DIR, exist_ok=True)
pytrends = TrendReq(hl="en-US", tz=0)

def pull_keyword(kw, timeframe, tag):
    """Pull one keyword alone. Returns (series or None, status text)."""
    path = f"{SAVE_DIR}/{tag}__{kw.replace(' ', '_')}.csv"
    if os.path.exists(path):                       # already pulled earlier, reuse it
        return pd.read_csv(path, index_col=0, parse_dates=True)[kw], "cached"
    for attempt in range(3):
        try:
            pytrends.build_payload([kw], timeframe=timeframe, geo=GEO)
            data = pytrends.interest_over_time()
            if data.empty:
                return None, "no data returned"
            data = data[~data["isPartial"].astype(bool)]    # drop the incomplete last week
            data[[kw]].to_csv(path)
            return data[kw], "ok"
        except Exception as e:
            print(f"    attempt {attempt + 1} failed ({type(e).__name__}), waiting 60 s")
            time.sleep(60)
    return None, "FAILED"

def pull_all(timeframe, tag):
    series, failed = {}, []
    for i, kw in enumerate(keywords, 1):
        s, status = pull_keyword(kw, timeframe, tag)
        extra = f" ({len(s)} weeks)" if s is not None else ""
        print(f"[{tag}] {i}/{len(keywords)}  {kw}: {status}{extra}")
        if s is None:
            failed.append(kw)
        else:
            series[kw] = s
        if status == "ok":
            time.sleep(SLEEP_SECONDS)
    print(f"[{tag}] finished: {len(series)} pulled, {len(failed)} failed {failed}\n")
    return pd.DataFrame(series), failed

recent, failed_recent = pull_all(RECENT_WINDOW, "recent")
old, failed_old = pull_all(OLD_WINDOW, "old")

If any keywords failed, wait a few minutes and **rerun the cell above**. Only failed keywords are re-requested.

Then verify both windows are weekly (7-day step). If the old window shows ~30-day steps, Google returned monthly data — shorten `OLD_WINDOW` slightly.


In [ ]:
for name, d in [("recent", recent), ("old", old)]:
    step = d.index.to_series().diff().median().days
    print(f"{name}: {d.index.min().date()} to {d.index.max().date()}, {len(d)} rows, step = {step} days")

## 3b. Stitch the two windows

Both windows are scaled to their own 0–100 range, so the older window is multiplied by a factor that makes its average match the recent window over the overlapping weeks. Keywords with very low volume in the overlap get an unreliable stitch factor and are listed separately.


In [ ]:
overlap = old.index.intersection(recent.index)
print(f"Overlap: {len(overlap)} weeks ({overlap.min().date()} to {overlap.max().date()})")

stitched, weak = {}, []
for kw in recent.columns:
    if kw not in old.columns:
        stitched[kw] = recent[kw]            # only the recent window is available
        continue
    o = old.loc[overlap, kw].mean()
    r = recent.loc[overlap, kw].mean()
    factor = r / o if (o > 0 and r > 0) else 1.0
    if o < 3 or r < 3:
        weak.append(kw)                      # too little volume in the overlap for a reliable factor
    older_part = old[kw][old.index < recent.index.min()] * factor
    stitched[kw] = pd.concat([older_part, recent[kw]])

df = pd.DataFrame(stitched)
df.to_csv("trends_weekly_raw.csv")          # save so we do not have to pull again
print(f"Stitched table: {df.shape[0]} weeks x {df.shape[1]} keywords")
print(f"Unreliable stitch factor (low volume in the overlap): {weak}")

## 3c. Stitch checks

Verify every keyword is present, then plot a few series with a vertical line at the seam to check the stitched part looks continuous.


In [ ]:
missing = [k for k in keywords if k not in df.columns]
print("Missing keywords:", missing)
df.index.min().date(), df.index.max().date(), df.shape

In [ ]:
seam = recent.index.min()
show = [k for k in ["cargo pants", "ballet flats", "blazer", "skinny jeans"] if k in df.columns] or list(df.columns[:4])

fig, axes = plt.subplots(len(show), 1, figsize=(12, 2.6 * len(show)), sharex=True)
axes = np.atleast_1d(axes)
for ax, kw in zip(axes, show):
    df[kw].plot(ax=ax)
    ax.axvline(seam, color="red", linestyle="--")      # where the two windows join
    ax.set_ylabel(kw)
axes[0].set_title("Stitched series (red line = start of the recent window)")
plt.show()

## 4. First look at the data

*(To skip the pull when reopening the notebook, load the saved file instead:)*
```python
df = pd.read_csv("trends_weekly_raw.csv", index_col=0, parse_dates=True)
```


In [ ]:
#df = pd.read_csv("trends_weekly_raw.csv", index_col=0, parse_dates=True)

In [ ]:
df.shape   # (number of weeks, number of keywords)

In [ ]:
df.index.min(), df.index.max()   # date range

In [ ]:
df.head()

In [ ]:
df.isna().sum().sum()   # total missing values

## 5. Summary statistics and zero-share filter

For each keyword: category, mean, std, max, peak-to-mean ratio, share of zero weeks, and unique non-zero values. Keywords with more than 30% zero weeks (`ZERO_CUTOFF`) are dropped — but note that emerging terms like `balletcore` and `gorpcore` have zeros because they barely existed before 2020, not because they are low-volume today (see open items in the README).


In [ ]:
summary = pd.DataFrame({
    "category": pd.Series(category),
    "mean": df.mean(),
    "std": df.std(),
    "max": df.max(),
    "peak_to_mean": df.max() / df.mean(),      # high = spiky keyword
    "zero_share": (df == 0).mean(),            # share of weeks with value 0
    "unique_values": df.nunique(),             # few unique values = coarse data
}).loc[df.columns].sort_values("zero_share", ascending=False)

summary.round(2)

Look at the `zero_share` column above and adjust `ZERO_CUTOFF` if needed before dropping keywords.


In [ ]:
ZERO_CUTOFF = 0.30

low_volume = list(summary[summary["zero_share"] > ZERO_CUTOFF].index)
df_clean = df.drop(columns=low_volume)
print(f"Dropped {len(low_volume)} low-volume keywords: {low_volume}")
df_clean.shape

### 6.2 Trends by category

Each keyword is divided by its own mean (so a value of 1 = average level for that keyword) and then averaged within each category. This puts all keywords on a comparable relative scale despite having different absolute ranges.


In [ ]:
norm = df_clean / df_clean.mean()          # each keyword divided by its own average

cat_series = norm.T.groupby(norm.columns.map(category)).mean().T

ax = cat_series.rolling(8).mean().plot(figsize=(12, 5), title="Average relative interest by category (8-week smoothing)")
ax.axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2020-06-30"), color="grey", alpha=0.2)   # COVID-19 lockdown period
plt.ylabel("Relative interest (1 = keyword's own average)")
plt.show()

### 6.3 Seasonality

Median across years of the category average by week of year. The median prevents one unusual year (e.g. the 2026 spike) from creating a fake seasonal peak.


In [ ]:
week_of_year = cat_series.index.isocalendar().week.values
cat_series.groupby(week_of_year).median().plot(figsize=(12, 5), title="Seasonality: median by week of year")
plt.xlabel("Week of year")
plt.ylabel("Relative interest")
plt.show()

In [ ]:
acf52 = df_clean.apply(lambda s: s.autocorr(lag=52)).sort_values(ascending=False)

n = min(10, len(acf52))
acf52.head(n).plot.barh(figsize=(8, 4), title="Most seasonal keywords (lag-52 autocorrelation)")
plt.gca().invert_yaxis()
plt.show()

### 6.4 Rising and falling keywords

Uses the **median** of the last 52 weeks vs. the 52 weeks before, so one spike week does not decide the result.


In [ ]:
last_year = df_clean.iloc[-52:].median()
prev_year = df_clean.iloc[-104:-52].median()
change = ((last_year / prev_year - 1) * 100).replace([np.inf, -np.inf], np.nan).dropna().sort_values()

n = min(10, len(change) // 2)
pd.concat([change.head(n), change.tail(n)]).plot.barh(figsize=(8, 7), title="Biggest falls and rises (median, last 52 weeks vs. the year before, %)")
plt.show()

### 6.5 Correlations

Correlation of **weekly changes** (not levels), so a shared long-term trend or one shared spike does not make every keyword look related. Change the keyword list to explore other categories.


In [ ]:
top = list(df_clean.columns[:20])

plt.figure(figsize=(10, 8))
sns.heatmap(df_clean[top].diff().corr(method="spearman"), cmap="coolwarm", center=0)
plt.title("Correlation of weekly changes (Spearman)")
plt.show()

### 6.6 What counts as "rising"?

Distribution of the next-4-week average divided by the last-4-week average. The threshold that defines "rising" is chosen from this distribution.


In [ ]:
past_4 = df_clean.rolling(4).mean()          # average of the last 4 weeks
future_4 = past_4.shift(-4)                  # average of the next 4 weeks

ratio = (future_4 / past_4).replace([np.inf, -np.inf], np.nan)
vals = ratio.stack().dropna()

vals.clip(0, 3).hist(bins=60, figsize=(8, 4))
plt.axvline(1, color="red", linestyle="--")
plt.title("Next-4-week average / last-4-week average")
plt.show()

thresholds = [1.1, 1.2, 1.3, 1.5]
pd.Series({f"> {t}": (vals > t).mean() for t in thresholds}, name="share of keyword-weeks").round(3)

## 7. Spring 2026 spike check

In the test run, unrelated categories all jumped in the same weeks. For each week compute the share of keywords that are more than 2× their rolling median — a high share suggests a market-wide event or data artifact, not a style trend.


In [ ]:
normal_level = df_clean.rolling(52, center=True, min_periods=26).median()
rel = (df_clean / normal_level).replace([np.inf, -np.inf], np.nan)     # value vs. the keyword's own normal level

share_spiking = (rel > 2).mean(axis=1)       # share of keywords at more than 2x normal in each week

share_spiking.plot(figsize=(12, 3), title="Share of keywords above 2x their normal level")
plt.show()
share_spiking.sort_values(ascending=False).head(10)

Flag weeks where a large share of keywords spike together (edit `SPIKE_SHARE` after looking at the plot). The flagged weeks are blanked and short gaps are filled by interpolation, producing a second "no-spike" version for sensitivity checks.


In [ ]:
SPIKE_SHARE = 0.5
spike_flag = share_spiking > SPIKE_SHARE
print("Flagged weeks:", list(spike_flag[spike_flag].index.strftime("%Y-%m-%d")))

df_nospike = df_clean.where(~spike_flag, np.nan, axis=0).interpolate(limit=3)   # blank flagged weeks, fill short gaps

## 8. Sensitivity: do results change without the spike?

Lag-52 autocorrelation with and without the flagged spike weeks. A big difference means the spike — not real seasonality — is driving the result.


In [ ]:
acf_compare = pd.DataFrame({
    "with spike weeks": df_clean.apply(lambda s: s.autocorr(lag=52)),
    "without spike weeks": df_nospike.apply(lambda s: s.autocorr(lag=52)),
})
acf_compare["difference"] = acf_compare["without spike weeks"] - acf_compare["with spike weeks"]
acf_compare.sort_values("difference").round(2)

## 9. Seasonality year by year
One line per year for a single keyword. If the lines roughly follow the same shape, the seasonality is real; if one year stands out, it is a one-off. Change `kw`.

In [ ]:
kw = "ballet flats" if "ballet flats" in df_clean.columns else df_clean.columns[0]
s = df_nospike[kw]

by_year = pd.DataFrame({"week": s.index.isocalendar().week.values,
                        "year": s.index.isocalendar().year.values,
                        "value": s.values})
by_year.pivot_table(index="week", columns="year", values="value").plot(figsize=(12, 5), title=f"{kw}: weekly interest by year")
plt.xlabel("Week of year")
plt.show()

## 10. Trend and seasonality decomposition
Splits one keyword into a long-term trend, a yearly seasonal pattern and the remainder (the `robust=True` option reduces the effect of spikes). A large remainder means the keyword is hard to predict from trend and season alone.

In [ ]:
from statsmodels.tsa.seasonal import STL

result = STL(df_nospike[kw].interpolate(), period=52, robust=True).fit()
result.plot()
plt.show()

# share of the variation left in the remainder (lower = more predictable from trend and season)
result.resid.var() / df_nospike[kw].interpolate().var()

## 11. Class balance of the "rising" label by year
If the share of rising weeks differs a lot between years, a random split would mislead us, and the chronological split must be checked for the same problem.

In [ ]:
years = vals.index.get_level_values(0).year
balance = pd.DataFrame({f"> {t}": (vals > t).groupby(years).mean() for t in [1.1, 1.2, 1.3]})
balance.round(3)

## 12. Is Google's data stable between pulls?
Pull three keywords again (ignoring the saved files) and compare with the first pull. A correlation close to 1 means sampling noise is small.

In [ ]:
for k in ["cargo pants", "ballet flats", "blazer"]:
    if k not in recent.columns:
        continue
    pytrends.build_payload([k], timeframe=RECENT_WINDOW, geo=GEO)
    again = pytrends.interest_over_time()[k]
    both = pd.concat([recent[k], again], axis=1).dropna()
    corr = both.corr().iloc[0, 1]
    diff = (both.iloc[:, 0] - both.iloc[:, 1]).abs().mean()
    print(f"{k}: correlation between pulls = {corr:.3f}, mean absolute difference = {diff:.2f}")
    time.sleep(SLEEP_SECONDS)

## 13. Save the cleaned data
Saves the cleaned data, the spike-adjusted version and the spike flags.

In [ ]:
df_clean.to_csv("trends_weekly_clean.csv")
df_nospike.to_csv("trends_weekly_nospike.csv")
spike_flag.to_csv("trends_spike_weeks.csv")

## Analysis of the Full-Pull Results (137 keywords, 2017-06 to 2026-10)

### 1. What we collected
- 489 weekly observations (2017-06-04 to 2026-10-11) for all 137 keywords. Both windows came back weekly (7-day step), and the 28-week overlap stitched without errors.
- There are 228 missing values in total. That equals the number of weeks before the recent window starts, so they come from a single keyword: `mob wife aesthetic`. Google returned no data for it in the older window, probably because it had almost no searches then. The notebook lists it as "failed", but nothing went wrong with the request.
- Several keywords have an unreliable stitch factor because their search volume in the overlap was very low (`barrel jeans`, `balloon skirt` and `oversized sunglasses` are visible in the printed list, which was cut off). The older part of these series is less trustworthy.

### 2. Data quality
- Resolution is much better than in the test run. The visible rows have 67-94 distinct values, not the coarse steps of about 3.4 we saw before, so pulling each keyword alone worked.
- The 30% zero-share cut-off dropped `balletcore` (53% zeros), `clean girl aesthetic` (50%) and `gorpcore` (31%). These are emerging aesthetics with almost no searches in the early years, so their zeros reflect when the term appeared, not low volume today. They are exactly the kind of trends we want to forecast, so this filter removes the wrong keywords. `quiet luxury` (28%) and `cottagecore` (27%) show the same pattern and only just survived.
- Some stitched series go above 100 (`cottagecore` reaches about 169). This means the older period was more popular than anything in the recent window, which is plausible, but it should be checked for low-volume keywords.
- In the seam plots, `cargo pants`, `ballet flats` and `skinny jeans` look continuous across the join. `blazer` appears to have a small step at the seam and deserves a closer look.

### 3. Trends
- `ballet flats` stayed at about 5-8 until 2022, rose from 2023 to about 35 in 2025, and reached about 65-90 in 2026. Among the four plotted keywords it is the only one that stays high after the spring 2026 spike, so it looks like a real, sustained revival.
- `cargo pants` rose steadily from about 8 in 2017 to about 50 in 2023-2024, then fell back to about 35 by 2025. `skinny jeans` declined slowly from about 20 to about 12 by 2025, consistent with a fading style. This also confirms that the +119% rise for `skinny jeans` in the test run was an artifact. `blazer` is comparatively flat (about 15-30).
- Most categories sit at about 0.5-1.0 times their own average until 2021 and drift up to about 1-1.5 by 2025. Every category jumps in spring 2026 (skirts and dresses to about 7 times average, jeans and denim about 5.5, shoes about 4) and then falls back. Aesthetics has an earlier bump of about 2.6 in early 2024.
- No obvious COVID break shows in the category averages (grey band). Individual keywords such as `leggings`, `joggers` and `sweatpants` have not been checked yet.

### 4. Seasonality
- Outerwear is the only clearly seasonal category: its median interest rises from about 0.55 in summer (weeks 22-28) to about 1.65 in mid-November (week 46). Other categories stay close to flat (about 0.7-1.1), with mild patterns such as skirts and dresses dipping at the end of the year.
- The lag-52 ranking is led by `mary janes`, `jorts`, `kitten heels`, `claw clip`, `ballet flats`, `clogs` and `baggy jeans` (about 0.93-0.96). Several of these are also the strongest long-term risers, so their values mostly reflect trend, not seasonality: a steadily rising series looks similar to itself a year earlier. `jorts`, `gingham` and `sage green` are more likely to be truly seasonal. We need a detrended measure before ranking keywords by seasonality.
- Removing the 4 flagged spike weeks changed the lag-52 values by only 0 to +0.06. With nine years of data, the spike has little effect on autocorrelation; trend matters more. This corrects my earlier concern from the 5-year test.
- For `ballet flats`, the yearly lines are roughly parallel at higher and higher levels, with a repeating bump around weeks 35-37 (early September) in 2023-2026. The STL decomposition shows a smooth rising trend, a seasonal swing that grows with the level (roughly +/-2 early on, +/-10-15 in recent years, so seasonality is multiplicative), and small residuals except in 2026. About 11% of the variance is left in the remainder, so trend and season explain most of its movement.

### 5. Rising and falling keywords
Median of the last 52 weeks compared with the 52 weeks before.

| Direction | Keywords (approximate change) |
| --- | --- |
| Biggest rises | `oversized sunglasses` +250%, `quiet luxury` +222%, `capsule wardrobe` +211%, `faux fur coat` +190%, `bermuda shorts` +165%, `capri pants` +148%, `polka dot` +143%, `ballet flats` +134%, `streetwear` +126%, `pleated trousers` +125% |
| Biggest falls | `old money style` -13%, `coquette` -12%, `bell bottoms` -6%, then `dark academia`, `belt bag` and `mom jeans` at about -3% |

- Rises of 125-250% against falls of at most 13% mean that nearly everything went up. Since this uses medians, it suggests most keywords were higher across the whole last year, not only in the spike weeks. A market-wide level shift (or a change in how Google scales the data) is mixed in with style-specific changes.
- Percentage changes on low-volume keywords are noisy. `oversized sunglasses` is also on the weak-stitch list.

### 6. Correlations (first 20 keywords: jeans and trousers only)
- Jeans styles move together (weekly-change correlations about 0.4-0.6). `leggings`, `joggers` and `sweatpants` form a second cluster (about 0.6-0.7 with each other) that also correlates with the slimmer jeans cuts.
- `jorts` and `bell bottoms` are the least related (about 0-0.2). For `jorts` this is likely due to strong summer seasonality.
- Even unrelated keywords have positive correlations (about 0.2-0.4), which points to a common market-wide factor, such as overall apparel searching or how Google samples and scales the data. A "market index" feature should help separate market-wide moves from style-specific ones.
- The heatmap covers only the first 20 keywords, so shoes, bags, accessories and aesthetics are not shown.

### 7. Spike weeks
- The share of keywords above 2x their normal level stays under about 0.1 for most of the 9 years (small yearly bumps, probably winter items). It then reaches 0.4-0.54 in several weeks of 2026: 2026-02-08 and 2026-02-15 (0.27 and 0.40), and 2026-03-29 to 2026-05-17 (0.28-0.54, highest on 2026-04-12).
- With the 0.5 threshold only 4 weeks were flagged (04-05, 04-12, 05-03, 05-10). Weeks that belong to the same event were not flagged, such as 03-29 (0.47), 04-19 (0.44), 04-26 (0.42) and 02-15 (0.40). The interpolated "no spike" version therefore still contains most of the spike.
- The cause is unknown. A simultaneous jump across unrelated categories is more likely a market-wide event or a data artifact than a style trend. To find out, check a few non-fashion control keywords, and a single-country region, for the same weeks.

### 8. Choosing the "rising" threshold
- The ratio histogram is centred on 1.0, bell-shaped with a mild right skew, and much smoother than in the test run, which had a spike at 1.0 caused by coarse data. A handful of extreme values sit at 0 and above 3.
- Share of keyword-weeks above each threshold: >1.1 is 24.1%, >1.2 is 11.9%, >1.3 is 6.9%, >1.5 is 2.9%. A threshold of 1.1 gives about 1 in 4 positives (reasonably balanced); 1.2 gives about 1 in 8.
- By year (2017-2022 visible): >1.1 is 20-27%, >1.2 is 9-13%, >1.3 is 5-7.5%. This is stable, with a higher share in 2020 (26.8%, 13.3%, 7.5%), probably because of COVID. The rows for 2023-2026 were cut off in the PDF and should be checked, since 2026 is likely higher because of the spike.
- A 10% change in a 4-week average may still be within normal noise. The stability check (section 12) shows how large that noise is.
- The "rising" label includes seasonal increases, such as outerwear every autumn. Decide whether that is wanted (it is a legitimate stocking signal) or whether to deseasonalise.

### 9. What to do next
1. **`mob wife aesthetic`:** fill its weeks before it appears with 0 (Google returned no data) or start its series at 2021-10, and log the choice.
2. **Zero-share filter:** measure zero share over the last 104 weeks (plus a minimum number of distinct values) and keep emerging terms. Re-include `balletcore`, `clean girl aesthetic` and `gorpcore` with an "emerging" flag.
3. **Spike handling:** lower `SPIKE_SHARE` to about 0.25 so the February and March-May weeks are all flagged. Do not interpolate long runs (the `limit=3` setting only fills short gaps anyway). Prefer excluding those weeks from labels and evaluation, or report results both ways. Check control keywords and a single-country region first.
4. **Market index:** add a market-wide index and relative-strength features, and redo the rise/fall list relative to the market.
5. **Seasonality:** use STL seasonal strength or a detrended autocorrelation, and consider working with logs or ratios because seasonality grows with the level.
6. **Stitching:** for weak-stitch keywords, use only the recent window or leave them out of long-history experiments. Look at the `blazer` seam.
7. **More checks:** extend the correlation heatmap to other categories and look at COVID effects on `leggings`, `joggers` and `sweatpants`.
8. **Before fixing thresholds:** review the 2023-2026 class balance and the section 12 stability output, then choose between 1.1 and 1.2.